# Spatial Durbin Models and Spatial Effects

The diagnostics from both the lag and the error model pointed the same way: the neighbors' covariates belong in the model. Adding them gives the spatial Durbin family, and it forces the question the lag model already raised, what a coefficient means when an outcome in one place depends on conditions in another. We answer it with direct and indirect effects.

In [ ]:
import arviz as az
import geopandas as gpd
import matplotlib.pyplot as plt
from libpysal.graph import Graph
from neighbayes.models import SAR, SDM, SLX

az.rcParams["stats.ci_prob"] = 0.95

In [ ]:
chi = gpd.read_file("data/Chicago-SDOH/Chi-SDOH.shp")
chi["ypll"] = chi["YPLL_rate"] / 1000  # premature mortality, in thousands of years of life lost
graph = Graph.build_contiguity(chi, rook=False).transform("r")

formula = "ypll ~ Blk14P + Hisp14P + HIS_ct"
fit_kwargs = dict(draws=1000, tune=1000, chains=4, random_seed=42)

## The SLX model

$$y = X\beta + WX\theta + \varepsilon$$

The SLX model adds the spatial lag of each covariate, the average of its values among a tract's neighbors. Each $\theta$ measures how much the neighbors' conditions matter for a tract's outcome, holding its own conditions fixed. The spillovers are local: a change in one tract reaches its neighbors and stops there.

In [ ]:
slx = SLX(formula=formula, data=chi, W=graph)
slx.fit(**fit_kwargs)
slx.summary(round_to=3)

All three lagged covariates matter, and two of them change the story. The coefficient on a tract's own Hispanic share is now close to zero, while the Hispanic share of its neighbors carries a negative coefficient: the protective association runs at a scale larger than the tract. And the neighbors' hardship matters more than a tract's own.

The diagnostics on the SLX model ask whether a spatial process remains once the lagged covariates are in.

In [ ]:
slx.spatial_diagnostics_decision()

Neither robust test rejects, and the tree keeps the SLX model.

## The spatial Durbin model

$$y = \rho W y + X\beta + WX\theta + \varepsilon$$

The spatial Durbin model, SDM, adds the lagged covariates to the spatial lag model, which is where the lag model's own diagnostics pointed.

In [ ]:
sdm = SDM(formula=formula, data=chi, W=graph)
sdm.fit(**fit_kwargs)
sdm.summary(round_to=3)

$\rho$ itself barely moves from the lag model's estimate, but its 95% interval now reaches down to zero. Once the neighbors' covariates enter, the data no longer require the outcome itself to spill over.

## Direct and indirect effects

In a model with $\rho W y$, the effect of a covariate is not a coefficient but a matrix. Solving the SDM for $y$,

$$y = (I - \rho W)^{-1}(X\beta + WX\theta + \varepsilon),$$

so the change in the outcome of every tract with respect to covariate $k$ in every tract is the $n \times n$ matrix

$$\frac{\partial y}{\partial x_k'} = (I - \rho W)^{-1}(\beta_k I + \theta_k W).$$

LeSage and Pace (2009) summarize it with three numbers. The direct effect is the average of its diagonal, the effect of a change in a tract on that same tract, including the feedback that returns through its neighbors. The indirect effect is the average row sum of its off-diagonal elements, the effect on a tract of a change in every other tract. The total effect is their sum. `neighbayes` computes all three for every posterior draw, so each effect has a posterior distribution and an interval.

In [ ]:
sdm.impacts()

Hardship's indirect effect is about twice its direct effect, and the Hispanic share's effect is entirely indirect. The percent Black population raises premature mortality within a tract, but its indirect effect is negative, so the total effect is about half the direct one.

Without $\rho$, the matrix reduces to $\beta_k I + \theta_k W$, and for a row-standardized graph the direct effects are the $\beta$ and the indirect effects are the $\theta$. The SLX model's effects are its coefficients.

In [ ]:
slx.impacts()

The SLX and SDM effects are nearly identical, which is what the SDM's small $\rho$ implies.

## What the lag model got wrong

The spatial lag model has effects too, but it computes them under a restriction. With no $\theta$, every covariate's effect matrix is a multiple of the same matrix, $(I - \rho W)^{-1}$, so the ratio of indirect to direct effect is the same for every covariate.

In [ ]:
sar = SAR(formula=formula, data=chi, W=graph)
sar.fit(**fit_kwargs)
sar_effects = sar.impacts()
sar_effects.assign(ratio=sar_effects["indirect"] / sar_effects["direct"])[
    ["direct", "indirect", "ratio"]
]

In the lag model, every indirect effect is about 14% of the direct effect, whatever the covariate. The data say otherwise: the indirect effect of the Black share runs opposite to its direct effect, and the indirect effect of hardship is twice its direct effect, not a seventh of it. The lag model fit the residual dependence well, but its account of how conditions in one tract affect another came from its functional form.

## The posterior of an effect

`impacts(return_posterior_samples=True)` returns the draws behind the table, one row per posterior draw and one column per covariate, so we can look at the whole distribution of an effect rather than its summary.

In [ ]:
summary, draws = sdm.impacts(return_posterior_samples=True)
hardship = list(summary.index).index("HIS_ct")

fig, ax = plt.subplots(figsize=(7, 4))
for kind in ["direct", "indirect", "total"]:
    ax.hist(draws[kind][:, hardship], bins=60, alpha=0.6, label=kind)
ax.set_xlabel("effect of the hardship index on premature mortality")
ax.legend();

The indirect effect sits well to the right of the direct effect, and the total to the right of both. A reader who took the coefficient on a tract's own hardship as its effect would understate the total effect by about two thirds.

<div class="alert alert-success" style="font-size:120%">
<b>Exercise</b>: <br>
The spatial error model's diagnostics pointed to the spatial Durbin error model, <code>SDEM</code>. Fit it, and compare its effects with those of the SLX and SDM models.
</div>

In [ ]:
# %load solutions/08.py